### Classical SVM on 1000 samples

In [4]:
import os
import ast
import numpy as np
import pandas as pd
import wfdb
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Path to your PTB-XL dataset
data_path = 'ptb-xl-1.0.3/'  # ← Set this ONCE

# Load metadata
df = pd.read_csv(os.path.join(data_path, 'ptbxl_database.csv'), index_col='ecg_id')
df.scp_codes = df.scp_codes.apply(lambda x: ast.literal_eval(x))

# Load SCP statements and filter for diagnostic classes
scp_df = pd.read_csv(os.path.join(data_path, 'scp_statements.csv'), index_col=0)
scp_diagnostic = scp_df[scp_df.diagnostic == 1]

# Map scp codes to diagnostic superclasses
def aggregate_diagnostic(y_dict):
    return list(set(
        scp_diagnostic.loc[key].diagnostic_class
        for key in y_dict.keys()
        if key in scp_diagnostic.index
    ))

df["diagnostic_superclass"] = df.scp_codes.apply(aggregate_diagnostic)

# Filter: keep clean normal or clearly abnormal cases (exclude STTC-only)
def filter_data(row):
    classes = row["diagnostic_superclass"]
    if not classes:
        return False
    if "NORM" in classes and len(classes) > 1:
        return False
    if "STTC" in classes and len(classes) == 1:
        return False
    return True

df = df[df.apply(filter_data, axis=1)]
df["is_normal"] = df["diagnostic_superclass"].apply(lambda x: "NORM" in x)

# Limit to smaller dataset for testing
df_short = df.head(1000)

# Load waveform data
def load_raw_data(df_subset, path):
    return np.array([
        wfdb.rdsamp(os.path.join(path, fname))[0]
        for fname in df_subset["filename_hr"]
    ])

X_raw = load_raw_data(df_short, data_path)  # shape: (N, 5000, 12)

# Extract 72 features (mean, std, min, max, median, peak-to-peak) per lead
def extract_features(X):
    means = X.mean(axis=1)
    stds = X.std(axis=1)
    mins = X.min(axis=1)
    maxs = X.max(axis=1)
    medians = np.median(X, axis=1)
    p2p = np.ptp(X, axis=1)  # fixed here

    return np.concatenate([means, stds, mins, maxs, medians, p2p], axis=1)

X_feat = extract_features(X_raw)
y = df_short["is_normal"].astype(int).values

# Train/test split
X_train, X_val, y_train, y_val = train_test_split(
    X_feat, y, test_size=0.2, random_state=42, stratify=y
)

# Train SVM
clf = SVC(kernel="rbf", probability=True)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_val)
print("Classification Report:\n", classification_report(y_val, y_pred, target_names=["abnormal", "normal"]))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))

Classification Report:
               precision    recall  f1-score   support

    abnormal       0.84      0.65      0.74        81
      normal       0.80      0.92      0.85       119

    accuracy                           0.81       200
   macro avg       0.82      0.79      0.79       200
weighted avg       0.81      0.81      0.80       200

Confusion Matrix:
 [[ 53  28]
 [ 10 109]]


In [6]:
import os, wfdb, ast
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# === Reload metadata and waveform ===
data_path = 'ptb-xl-1.0.3/'

df = pd.read_csv(os.path.join(data_path, 'ptbxl_database.csv'), index_col='ecg_id')
df.scp_codes = df.scp_codes.apply(ast.literal_eval)
scp_df = pd.read_csv(os.path.join(data_path, 'scp_statements.csv'), index_col=0)
scp_diagnostic = scp_df[scp_df.diagnostic == 1]

def aggregate_diagnostic(y_dict):
    return list(set(
        scp_diagnostic.loc[key].diagnostic_class
        for key in y_dict.keys() if key in scp_diagnostic.index
    ))

df["diagnostic_superclass"] = df.scp_codes.apply(aggregate_diagnostic)

def filter_data(row):
    classes = row["diagnostic_superclass"]
    if not classes:
        return False
    if "NORM" in classes and len(classes) > 1:
        return False
    if "STTC" in classes and len(classes) == 1:
        return False
    return True

df = df[df.apply(filter_data, axis=1)]
df["is_normal"] = df["diagnostic_superclass"].apply(lambda x: "NORM" in x)

df_short = df.head(1000)

def load_raw_data(df_subset, path):
    return np.array([
        wfdb.rdsamp(os.path.join(path, fname))[0]
        for fname in df_subset["filename_hr"]
    ])

X_raw = load_raw_data(df_short, data_path)
y = df_short["is_normal"].astype(int).values
X_train_raw, X_val_raw, y_train, y_val = train_test_split(X_raw, y, test_size=0.2, random_state=42, stratify=y)

# === Re-extract features (72 features) ===
def extract_features(X):
    means = X.mean(axis=1)
    stds = X.std(axis=1)
    mins = X.min(axis=1)
    maxs = X.max(axis=1)
    medians = np.median(X, axis=1)
    p2p = np.ptp(X, axis=1)
    return np.concatenate([means, stds, mins, maxs, medians, p2p], axis=1)

X_train_feats = extract_features(X_train_raw)
X_val_feats = extract_features(X_val_raw)

print("✅ Feature extraction complete.")
print("Train shape:", X_train_feats.shape)
print("Val shape:", X_val_feats.shape)

✅ Feature extraction complete.
Train shape: (800, 72)
Val shape: (200, 72)


### Autoencoder to compress data

In [8]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

# Device
device = torch.device("cpu")

# Scale flattened features before feeding into AE
scaler_ae = MinMaxScaler()
X_train_feats = scaler_ae.fit_transform(X_train_feats)
X_val_feats = scaler_ae.transform(X_val_feats)

# Convert numpy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train_feats, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_feats, dtype=torch.float32)

# Autoencoder model
class Autoencoder(nn.Module):
    def __init__(self, input_dim=72, latent_dim=20):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 36),
            nn.ReLU(),
            nn.Linear(36, latent_dim)  # <-- more than 6, still within QSVM range
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 36),
            nn.ReLU(),
            nn.Linear(36, input_dim)
        )
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

model = Autoencoder().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

n_epochs = 100
batch_size = 64

# Dataloaders
train_loader = DataLoader(TensorDataset(X_train_tensor), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor), batch_size=batch_size)

# Training loop
for epoch in range(n_epochs):
    model.train()
    train_loss = 0
    for batch in train_loader:
        x = batch[0].to(device)
        optimizer.zero_grad()
        x_hat, _ = model(x)
        loss = loss_fn(x_hat, x)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation loss
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            x = batch[0].to(device)
            x_hat, _ = model(x)
            loss = loss_fn(x_hat, x)
            val_loss += loss.item() * x.size(0)

    val_loss /= len(val_loader.dataset)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Train Loss = {train_loss:.6f}, Val Loss = {val_loss:.6f}")

# Encode features
model.eval()
with torch.no_grad():
    _, X_train_encoded = model(X_train_tensor.to(device))
    _, X_val_encoded = model(X_val_tensor.to(device))

X_train_encoded = X_train_encoded.cpu().numpy()
X_val_encoded = X_val_encoded.cpu().numpy()

print("✅ Autoencoder finished. Encoded shape:", X_train_encoded.shape)

Epoch 0: Train Loss = 0.227781, Val Loss = 0.197404
Epoch 10: Train Loss = 0.011618, Val Loss = 0.011520
Epoch 20: Train Loss = 0.008825, Val Loss = 0.008297
Epoch 30: Train Loss = 0.006767, Val Loss = 0.006524
Epoch 40: Train Loss = 0.006218, Val Loss = 0.005930
Epoch 50: Train Loss = 0.005571, Val Loss = 0.005174
Epoch 60: Train Loss = 0.005058, Val Loss = 0.004685
Epoch 70: Train Loss = 0.004591, Val Loss = 0.004308
Epoch 80: Train Loss = 0.004197, Val Loss = 0.003965
Epoch 90: Train Loss = 0.003854, Val Loss = 0.003602
✅ Autoencoder finished. Encoded shape: (800, 20)


### Classical SVM on Autoencoded data

In [9]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Train classical SVM
clf_ae = SVC(kernel="rbf", probability=True)
clf_ae.fit(X_train_encoded, y_train)

# Predict and evaluate
y_pred_ae = clf_ae.predict(X_val_encoded)

print("🎯 Classification Report (Autoencoder features + Classical SVM):\n")
print(classification_report(y_val, y_pred_ae, target_names=["abnormal", "normal"]))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred_ae))

🎯 Classification Report (Autoencoder features + Classical SVM):

              precision    recall  f1-score   support

    abnormal       0.85      0.49      0.62        81
      normal       0.73      0.94      0.82       119

    accuracy                           0.76       200
   macro avg       0.79      0.72      0.72       200
weighted avg       0.78      0.76      0.74       200

Confusion Matrix:
 [[ 40  41]
 [  7 112]]


### QSVM on Autoencoded Data with smaller size

In [10]:
import time
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
import pennylane as qml

# Use best 6 autoencoded features
n_qubits = 6
variances = np.var(X_train_encoded, axis=0)
top_features_idx = np.argsort(variances)[::-1][:6]
X_train_q = X_train_encoded[:, top_features_idx]
X_val_q = X_val_encoded[:, top_features_idx]

# Scale to [0, π] for angle embedding
scaler = MinMaxScaler((0, np.pi))
scaler.fit(np.vstack([X_train_q, X_val_q]))
X_train_q = scaler.transform(X_train_q)
X_val_q = scaler.transform(X_val_q)

# Subsample for speed
n_train_samples = 300
X_train_q_small = X_train_q[:n_train_samples]
y_train_small = y_train[:n_train_samples]

# Define kernel circuit
def angle_embedding_rz_ry(x, wires):
    for i in range(len(x)):
        qml.RY(x[i], wires=wires[i])
        qml.RZ(x[i], wires=wires[i])


class QuantumKernel:
    def __init__(self, n_qubits):
        self.n_qubits = n_qubits
        self.wires = list(range(n_qubits))
        self.dev = qml.device("default.qubit", wires=n_qubits)  # analytic mode

        @qml.qnode(self.dev)
        def kernel_circuit(x1, x2):
            angle_embedding_rz_ry(x1, self.wires)
            qml.adjoint(angle_embedding_rz_ry)(x2, self.wires)
            return qml.probs(wires=self.wires)

        self.qnode = kernel_circuit

    def __call__(self, X1, X2):
        N1, N2 = X1.shape[0], X2.shape[0]
        K = np.zeros((N1, N2))
        for i in tqdm(range(N1), desc="Computing kernel matrix"):
            for j in range(N2):
                probs = self.qnode(X1[i], X2[j])
                K[i, j] = probs[0]  # All-zero probability
        return K
      

# Run QSVM
qkernel = QuantumKernel(n_qubits=n_qubits)
start_time = time.time()

K_train = qkernel(X_train_q_small, X_train_q_small)
K_val = qkernel(X_val_q, X_train_q_small)

clf_q = SVC(kernel="precomputed")
clf_q.fit(K_train, y_train_small)
y_pred_q = clf_q.predict(K_val)

elapsed = time.time() - start_time
print("\n🎯 QSVM Results (Autoencoded Features, 6 qubits):")
print(classification_report(y_val, y_pred_q, target_names=["abnormal", "normal"]))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred_q))
print(f"⏱️ Time elapsed: {elapsed:.2f} seconds")

Computing kernel matrix: 100%|██████████| 200/200 [02:44<00:00,  1.21it/s]


🎯 QSVM Results (Autoencoded Features, 6 qubits):
              precision    recall  f1-score   support

    abnormal       0.79      0.51      0.62        81
      normal       0.73      0.91      0.81       119

    accuracy                           0.74       200
   macro avg       0.76      0.71      0.71       200
weighted avg       0.75      0.74      0.73       200

Confusion Matrix:
 [[ 41  40]
 [ 11 108]]
⏱️ Time elapsed: 448.09 seconds
